# Pattern 3: Model Registry sync + inference specification logging

Like pattern 2, the **SageMaker AI Model Registry** is the governance hub — but
instead of repacking the model, we log an **inference specification** against the
MLflow model with the `sagemaker-mlflow` plugin (`>= 0.5.0`). When
`mlflow.register_model()` triggers the sync, the Model Package is created **already
deployable**, serving the model **directly from the MLflow artifact store** via an
`S3Prefix` ModelDataSource — no repacking, no artifact copies.

```
MLflow artifact store  ◀────────── serves directly from ──────────┐
   │ log_inference_specification()                                │
   ▼                                                              │
MLflow ──register──▶ auto-sync ──▶ Model Package (deployable) ──▶ Endpoint
```

**When to choose this pattern:**
- you want the MLflow artifact store to remain the **single source of truth** for the
  served model bytes
- you (or your platform team) are comfortable providing the framework container image
  and, where needed, a small inference script

**Trade-offs:**
- the specification is **your responsibility**: container image, environment
  variables, and — for containers without a default model loader — an
  `inference.py` uploaded into the artifact store
- a few SageMaker-specific conventions to know (`SAGEMAKER_PROGRAM`,
  `SAGEMAKER_SUBMIT_DIRECTORY`, `/opt/ml/model/code/`)

> **Order matters:** the spec must be logged **before** `mlflow.register_model()`,
> because the sync copies it onto the Model Package at registration time.

In [ ]:
%store -r mlflow_app_arn
%store -r mlflow_model_ids
%store -r model_base_name
%store -r execution_role
%store -r region

import json
import time

import boto3
import mlflow
import sagemaker_mlflow

mlflow.set_tracking_uri(mlflow_app_arn)
mlflow_client = mlflow.MlflowClient()

# This pattern works exclusively with its own logged model — the inference
# specification logged below attaches to it, leaving the other patterns'
# logged models untouched.
mlflow_model_id = mlflow_model_ids["infspec"]

## Step 1: Resolve the serving container image

We serve with the SageMaker **SKLearn framework container**, resolved for the current
region with the SDK v3 `image_uris` helper — no hardcoded per-region ECR URIs.

In [ ]:
from sagemaker.core import image_uris

sklearn_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.4-2-py312",
    image_scope="inference",
    instance_type="ml.m5.xlarge",
)
print(f"Inference container image: {sklearn_image}")

## Step 2: Log a minimal `inference.py` as a model artifact

The SKLearn serving container has **no default model loader** — it requires a user
inference script referenced by the `SAGEMAKER_PROGRAM` environment variable (this is
what `SKLearnModel.deploy()` normally wires up by repacking the model tarball).

Since we deploy straight from the Model Package instead, we log a minimal
`code/inference.py` with a `model_fn` **as an artifact of the logged model** using
`MlflowClient.log_model_artifacts()`. The `S3Prefix` ModelDataSource downloads
everything under the model's artifact location, so the script lands at
`/opt/ml/model/code/inference.py` on the endpoint. Without it, the endpoint fails
every `/ping` health check and never reaches `InService`.

Logging through MLflow (rather than a raw S3 upload) keeps the artifact store as the
single source of truth — the script is visible in the MLflow UI, and the upload goes
through the MLflow app's own artifact access mechanism, with no direct
`s3:PutObject` permission on the artifact bucket required.

> This works because `00_setup_and_train.ipynb` logged the model with
> `serialization_format="pickle"` — a plain `model.pkl` the script can load.

In [ ]:
import os
import tempfile

logged_model = mlflow_client.get_logged_model(mlflow_model_id)

INFERENCE_SCRIPT = """\
import os
import pickle


def model_fn(model_dir):
    with open(os.path.join(model_dir, "model.pkl"), "rb") as f:
        return pickle.load(f)
"""

# Log through MLflow against the *logged model* (not the run): the file must land
# under logged_model.artifact_location, which is where the S3Prefix ModelDataSource
# points. log_model_artifacts preserves the local directory structure, so
# code/inference.py arrives at <artifact_location>/code/inference.py.
with tempfile.TemporaryDirectory() as tmp:
    os.makedirs(os.path.join(tmp, "code"))
    with open(os.path.join(tmp, "code", "inference.py"), "w") as f:
        f.write(INFERENCE_SCRIPT)
    mlflow_client.log_model_artifacts(mlflow_model_id, tmp)

print(f"Logged code/inference.py to {logged_model.artifact_location}/code/")

## Step 3: Log the inference specification

`sagemaker_mlflow.log_inference_specification()` attaches the container image, the
`S3Prefix` ModelDataSource pointing at the MLflow artifact location, and the
environment variables that point the framework container at the inference script.

In [ ]:
inference_spec = {
    "Containers": [{
        "Image": sklearn_image,
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": logged_model.artifact_location + "/",
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
        # Point the framework container at the inference script; without these
        # the sklearn container fails every /ping health check.
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        },
    }],
    "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.xlarge"],
}

sagemaker_mlflow.log_inference_specification(
    mlflow_model_id, inference_specification=inference_spec
)
print("Inference specification logged on the MLflow model.")

The logged model's artifact view in the MLflow UI now shows everything the endpoint
will serve from, in one place: `model.pkl`, `code/inference.py`, and the logged
`sagemaker_inference_specification.json`:

![The MLflow artifact browser showing model.pkl, code/inference.py, and sagemaker_inference_specification.json together](img/mlflow-inference-spec.png)

## Step 4: Register — auto-sync creates a *deployable* Model Package

Because the spec was logged first, the synced Model Package arrives with its
`InferenceSpecification` filled — no post-registration `update_model_package` needed.

In [ ]:
registered_name = f"{model_base_name}-infspec"

mv = mlflow.register_model(f"models:/{mlflow_model_id}", registered_name)
print(f"Registered {registered_name} v{mv.version}")

# The sync runs asynchronously: poll the model version until the tag appears.
sm_model_package_arn = None
for _ in range(20):
    mv_get = mlflow_client.get_model_version(registered_name, mv.version)
    sm_model_package_arn = mv_get.tags.get("sagemaker.model_package_arn")
    if sm_model_package_arn:
        break
    print(".", end="", flush=True)
    time.sleep(3)

if not sm_model_package_arn:
    raise TimeoutError("SageMaker Model Package was not auto-created within timeout")
print(f"\nSynced Model Package: {sm_model_package_arn}")

## Step 5: Inspect, approve, and deploy from the Model Registry

Registry-driven deployment with SDK v3 typed resources — identical to pattern 2.

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig, Model, ModelPackage
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

model_package = ModelPackage.get(model_package_name=sm_model_package_arn)
# Workaround: DescribeModelPackage does not return ModelPackageName for versioned
# packages, which breaks refresh()/update(). The API accepts an ARN in that field,
# so backfill it.
model_package.model_package_name = model_package.model_package_arn

print("Has inference specification:", model_package.inference_specification is not None)

model_package.update(model_approval_status="Approved")
print("Model Package approved.")

In [ ]:
suffix = time.strftime("%Y%m%d-%H%M%S")
resource_name = f"{model_base_name}-is-{suffix}"

deployed_model = Model.create(
    model_name=resource_name,
    primary_container=ContainerDefinition(model_package_name=sm_model_package_arn),
    execution_role_arn=execution_role,
)
endpoint_config = EndpointConfig.create(
    endpoint_config_name=resource_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=resource_name,
            initial_instance_count=1,
            instance_type="ml.m5.xlarge",
        )
    ],
)
endpoint = Endpoint.create(
    endpoint_name=resource_name,
    endpoint_config_name=resource_name,
)

endpoint_name = endpoint.endpoint_name
print(f"Creating endpoint {resource_name} (takes a few minutes)...")

In [ ]:
sm_client = boto3.client("sagemaker")
print(f"Polling endpoint: {endpoint_name}")

terminal_states = {"InService", "Failed"}
while True:
    desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "InService":
    raise RuntimeError(
        f"Endpoint {endpoint_name} deployment ended with status '{status}': "
        f"{desc.get('FailureReason')}"
    )
print(f"Endpoint {endpoint_name} is InService.")

# Re-obtain the Endpoint resource for downstream .invoke() calls
endpoint = Endpoint.get(endpoint_name=endpoint_name)

In [ ]:
# The SKLearn serving container follows the SageMaker Scikit-learn input conventions;
# invoke with a CSV payload matching the four training features.
response = endpoint.invoke(
    body="0.5,-1.2,0.3,0.8\n1.1,0.4,-0.7,0.2",
    content_type="text/csv",
    accept="application/json",
)
print("Prediction:", response.body.read().decode())

## What you get — and what you don't

✅ Model Package is **born deployable** — no post-sync mutation, no repacking
✅ The endpoint serves **directly from the MLflow artifact store** — single source of
   truth for the model bytes
✅ Clean fit for governed promotion flows: register → review → approve → deploy

❌ You own the specification: container image, environment variables, and (for
   containers without a default loader) the `inference.py` upload
❌ The `/opt/ml/model/code/` + `SAGEMAKER_PROGRAM` conventions are not obvious if you
   haven't worked with SageMaker framework containers before

**Pattern 2 vs pattern 3 in one line:** `ModelBuilder` writes the inference code for
you but copies the model out of MLflow; inference spec logging keeps MLflow as the
source of truth but the inference code is on you.

## Teardown (and run `04_cleanup.ipynb` at the end)

The cells below use the resource objects created earlier in this notebook — if the
kernel restarted since deployment, skip them and run `04_cleanup.ipynb` instead.

In [ ]:
# endpoint.delete()
# endpoint_config.delete()
# deployed_model.delete()